In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.over_sampling import RandomOverSampler


def run_rf_model(x_train_path, x_test_path, y_train_path, y_test_path, upsample=False):
    # 1. read data
    X_train = pd.read_csv(x_train_path)
    X_test = pd.read_csv(x_test_path)
    y_train = pd.read_csv(y_train_path)["y"]
    y_test = pd.read_csv(y_test_path)["y"]

    # 2. define CV setting
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # 3. define parameter grid
    param_grid = {
        "max_depth": [None, 5, 10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None]
    }

    # 4. tune model on training data
    if upsample:
        upsampler = RandomOverSampler(random_state=42)
        X_train_tune, y_train_tune = upsampler.fit_resample(
            X_train,
            y_train
        )
    else:
        X_train_tune, y_train_tune = X_train, y_train

    rf_base = RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        n_jobs=-1
    )

    rf_search = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )

    rf_search.fit(X_train_tune, y_train_tune)
    best_params = rf_search.best_params_

    # 5. define tuned model
    rf_model = RandomForestClassifier(
        n_estimators=250,
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        max_features=best_params["max_features"],
        random_state=42,
        n_jobs=-1
    )

    # 6. fivefold cross-validation with tuned model
    cv_scores = {
        "AUC": [],
        "Precision": [],
        "Recall": [],
        "Accuracy": [],
        "F1": []
    }

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]

        X_val_fold = X_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]

        if upsample:
            # Apply random over-sampling only to the training fold
            upsampler = RandomOverSampler(random_state=42)
            X_train_fold, y_train_fold = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

        rf_model.fit(X_train_fold, y_train_fold)

        y_val_pred = rf_model.predict(X_val_fold)
        y_val_prob = rf_model.predict_proba(X_val_fold)[:, 1]

        cv_scores["AUC"].append(
            roc_auc_score(y_val_fold, y_val_prob)
        )
        cv_scores["Precision"].append(
            precision_score(y_val_fold, y_val_pred, zero_division=0)
        )
        cv_scores["Recall"].append(
            recall_score(y_val_fold, y_val_pred, zero_division=0)
        )
        cv_scores["Accuracy"].append(
            accuracy_score(y_val_fold, y_val_pred)
        )
        cv_scores["F1"].append(
            f1_score(y_val_fold, y_val_pred, zero_division=0)
        )

    cv_metrics_df = pd.DataFrame({
        "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
        "Mean CV Score": [
            np.mean(cv_scores["AUC"]),
            np.mean(cv_scores["Precision"]),
            np.mean(cv_scores["Recall"]),
            np.mean(cv_scores["Accuracy"]),
            np.mean(cv_scores["F1"])
        ],
        "CV Std": [
            np.std(cv_scores["AUC"], ddof=1),
            np.std(cv_scores["Precision"], ddof=1),
            np.std(cv_scores["Recall"], ddof=1),
            np.std(cv_scores["Accuracy"], ddof=1),
            np.std(cv_scores["F1"], ddof=1)
        ]
    }).round(3)

    # 7. fit final model on full training data
    if upsample:
        upsampler = RandomOverSampler(random_state=42)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train,
            y_train
        )
        rf_model.fit(X_train_final, y_train_final)
    else:
        rf_model.fit(X_train, y_train)

    # 8. predict on independent test set
    y_pred = rf_model.predict(X_test)
    y_prob = rf_model.predict_proba(X_test)[:, 1]

    # 9. test set metrics
    test_metrics_df = pd.DataFrame({
        "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
        "Test Score": [
            roc_auc_score(y_test, y_prob),
            precision_score(y_test, y_pred, zero_division=0),
            recall_score(y_test, y_pred, zero_division=0),
            accuracy_score(y_test, y_pred),
            f1_score(y_test, y_pred, zero_division=0)
        ]
    }).round(3)

    return {
        "best_params": best_params,
        "cv_metrics": cv_metrics_df,
        "test_metrics": test_metrics_df
    }

In [3]:
def make_cv_summary(results_dict):
    rows = []

    for model_name, result in results_dict.items():
        cv_df = result["cv_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            mean_value = cv_df.loc[
                cv_df["Metric"] == metric,
                "Mean CV Score"
            ].values[0]

            cv_std = cv_df.loc[
                cv_df["Metric"] == metric,
                "CV Std"
            ].values[0]

            # Standard error across five folds
            cv_se = cv_std / np.sqrt(5)

            row[metric] = f"{mean_value:.3f} ({cv_se:.3f})"

        rows.append(row)

    return pd.DataFrame(rows)


def make_test_summary(results_dict):
    rows = []

    for model_name, result in results_dict.items():
        test_df = result["test_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            test_value = test_df.loc[
                test_df["Metric"] == metric,
                "Test Score"
            ].values[0]

            row[metric] = round(test_value, 3)

        rows.append(row)

    return pd.DataFrame(rows)

In [4]:
# Run random forest models for Model 1
result_1 = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_1.csv",
    x_test_path="../data/model_inputs/X_test_model_1.csv",
    y_train_path="../data/model_inputs/y_train_model_1.csv",
    y_test_path="../data/model_inputs/y_test_model_1.csv",
    upsample=False
)

result_1a = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_1a.csv",
    x_test_path="../data/model_inputs/X_test_model_1a.csv",
    y_train_path="../data/model_inputs/y_train_model_1a.csv",
    y_test_path="../data/model_inputs/y_test_model_1a.csv",
    upsample=True
)

result_1b = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_1b.csv",
    x_test_path="../data/model_inputs/X_test_model_1b.csv",
    y_train_path="../data/model_inputs/y_train_model_1b.csv",
    y_test_path="../data/model_inputs/y_test_model_1b.csv",
    upsample=True
)

In [5]:
# Run random forest models for Model 2
result_2 = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_2.csv",
    x_test_path="../data/model_inputs/X_test_model_2.csv",
    y_train_path="../data/model_inputs/y_train_model_2.csv",
    y_test_path="../data/model_inputs/y_test_model_2.csv",
    upsample=False
)

result_2a = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_2a.csv",
    x_test_path="../data/model_inputs/X_test_model_2a.csv",
    y_train_path="../data/model_inputs/y_train_model_2a.csv",
    y_test_path="../data/model_inputs/y_test_model_2a.csv",
    upsample=True
)

result_2b = run_rf_model(
    x_train_path="../data/model_inputs/X_train_model_2b.csv",
    x_test_path="../data/model_inputs/X_test_model_2b.csv",
    y_train_path="../data/model_inputs/y_train_model_2b.csv",
    y_test_path="../data/model_inputs/y_test_model_2b.csv",
    upsample=True
)

In [6]:
# Store Model 1 results
rf_model1_results = {
    "RF Model 1": result_1,
    "RF Model 1a": result_1a,
    "RF Model 1b": result_1b
}

# Store Model 2 results
rf_model2_results = {
    "RF Model 2": result_2,
    "RF Model 2a": result_2a,
    "RF Model 2b": result_2b
}


# Print best parameters
print("\n===== Best Parameters (rf model 1) =====")
for model_name, result in rf_model1_results.items():
    print(f"\n{model_name}")
    print(result["best_params"])

print("\n===== Best Parameters (rf model 2) =====")
for model_name, result in rf_model2_results.items():
    print(f"\n{model_name}")
    print(result["best_params"])


===== Best Parameters (rf model 1) =====

RF Model 1
{'max_depth': None, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2}

RF Model 1a
{'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2}

RF Model 1b
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}

===== Best Parameters (rf model 2) =====

RF Model 2
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2}

RF Model 2a
{'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2}

RF Model 2b
{'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2}


In [9]:
# Create summary tables for Model 1
cv_summary_model1_df = make_cv_summary(rf_model1_results)
test_summary_model1_df = make_test_summary(rf_model1_results)

print("\n===== CV Summary Table (rf model 1) =====")
print(cv_summary_model1_df.to_string(index=False))

print("\n===== Test Summary Table (rf model 1) =====")
print(test_summary_model1_df.to_string(index=False))

cv_summary_model1_df.to_csv("../results/rf_model1_cv_summary.csv", index=False)
test_summary_model1_df.to_csv("../results/rf_model1_test_summary.csv", index=False)


===== CV Summary Table (rf model 1) =====
      Model           AUC     Precision        Recall      Accuracy            F1
 RF Model 1 0.912 (0.001) 0.853 (0.002) 0.854 (0.003) 0.842 (0.000) 0.853 (0.000)
RF Model 1a 0.853 (0.006) 0.696 (0.011) 0.507 (0.008) 0.816 (0.003) 0.586 (0.004)
RF Model 1b 0.886 (0.001) 0.873 (0.000) 0.912 (0.001) 0.844 (0.001) 0.892 (0.000)

===== Test Summary Table (rf model 1) =====
      Model   AUC  Precision  Recall  Accuracy    F1
 RF Model 1 0.907      0.843   0.855     0.836 0.849
RF Model 1a 0.856      0.691   0.530     0.815 0.600
RF Model 1b 0.883      0.873   0.911     0.843 0.891


In [10]:
# Create summary tables for Model 2
cv_summary_model2_df = make_cv_summary(rf_model2_results)
test_summary_model2_df = make_test_summary(rf_model2_results)

print("\n===== CV Summary Table (rf model 2) =====")
print(cv_summary_model2_df.to_string(index=False))

print("\n===== Test Summary Table (rf model 2) =====")
print(test_summary_model2_df.to_string(index=False))

cv_summary_model2_df.to_csv("../results/rf_model2_cv_summary.csv", index=False)
test_summary_model2_df.to_csv("../results/rf_model2_test_summary.csv", index=False)


===== CV Summary Table (rf model 2) =====
      Model           AUC     Precision        Recall      Accuracy            F1
 RF Model 2 0.830 (0.001) 0.783 (0.002) 0.883 (0.002) 0.765 (0.001) 0.830 (0.000)
RF Model 2a 0.780 (0.002) 0.716 (0.004) 0.735 (0.004) 0.706 (0.003) 0.725 (0.003)
RF Model 2b 0.834 (0.002) 0.835 (0.002) 0.894 (0.003) 0.795 (0.003) 0.863 (0.002)

===== Test Summary Table (rf model 2) =====
      Model   AUC  Precision  Recall  Accuracy    F1
 RF Model 2 0.830      0.779   0.885     0.761 0.828
RF Model 2a 0.788      0.723   0.749     0.713 0.736
RF Model 2b 0.834      0.828   0.898     0.792 0.861
